In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(np.__version__)
print(pd.__version__)

读取cluster 7*7的数据

In [ ]:
df = pd.read_csv("clusters_7x7_all_layers.csv")
gMaxAdc = df.max().max()
print("Global Max ADC =", gMaxAdc)  #找到最大的ADC
print(df.shape)
df.head()

查看列名

In [ ]:
print(df.columns.tolist())

读取第一个Patch

In [ ]:
# cluster size  =  WINDOW_SIZE^2
WINDOW_SIZE = 7
SKIP_COLUMN = 6
matrix_size = WINDOW_SIZE*WINDOW_SIZE
patch_values = df.iloc[0, SKIP_COLUMN:matrix_size+SKIP_COLUMN].values

patch = patch_values.reshape(WINDOW_SIZE,WINDOW_SIZE)

print(patch)

显示一个cell

In [ ]:
plt.figure(figsize=(5,5))

plt.imshow(
    patch,
    origin="lower"
)

plt.colorbar()

plt.title("Cluster 0")

plt.show()

显示多个cell

In [ ]:
fig, axes = plt.subplots(
    3, 2,
    figsize=(8,12)
)

axes = axes.flatten()

for n in range(6):

    patch = (
        df.iloc[n, SKIP_COLUMN:matrix_size+SKIP_COLUMN]
        .values
        .reshape(WINDOW_SIZE,WINDOW_SIZE)
    )

    im = axes[n].imshow(
        patch,
        origin="lower"
    )

    axes[n].set_title(
        f"Cluster {n}"
    )

fig.colorbar(
    im,
    ax=axes
)

#plt.tight_layout()
plt.show()

定义计算特征值的子函数
Total ADC=∑ADCi​
max_adc 找出最大值
center_adc在7*7的矩阵里面找中间3*3的
nonzero_count 有多少个 count不是0
occupancy：非0像素的占比occupancy = nonzero_count / 49
    Low Occupancy：非0像素少
    Dense：非0像素多
edge_adc：外面24个格子的和
    X X X X X X X
    X           X
    X           X
    X           X
    X           X
    X           X
    X X X X X X X
edge_ratio：边缘能力的占比：edge_ratio = edge_adc / total_adc

if occupancy < 0.1:
    Low Occupancy

elif edge_ratio > 0.25:
    Edge Heavy

elif max_adc > 3000:
    High ADC

elif occupancy > 0.4:
    Dense

In [ ]:
def patch_features(patch):

    total_adc = np.sum(patch)  #把49个格子全部加起来，表示cluster的总能量，物理意义：这个Cluster总共沉积了多少能量
    max_adc = np.max(patch)    #找最大ADC，物理意义：peaking有多高
    center_adc = patch[3, 3]    #中心坐标：(3,3)  center_adc ≈ max_adc 验证peaking是不是放在中间

    nonzero_count = np.sum(patch > 0)    # 有多少个格子不是0
    occupancy = nonzero_count / 49    #物理意义这个cluster有多拥挤

    #做一个7*7全是FALSE的矩阵，把边缘填TRUE，表示只看边缘
    edge_mask = np.zeros_like(patch, dtype=bool)  
    edge_mask[0, :] = True
    edge_mask[6, :] = True
    edge_mask[:, 0] = True
    edge_mask[:, 6] = True

    edge_adc = np.sum(patch[edge_mask])
    edge_nonzero_count = np.sum(patch[edge_mask] > 0)  #看边缘24个格子，有几个有信号，物理意义：边界拥挤程度
    edge_occupancy = edge_nonzero_count / 24

    if total_adc > 0:
        edge_ratio = edge_adc / total_adc
    else:
        edge_ratio = 0

    #Prominence:中心值和周围值对比 
    neighbors = patch[2:5, 2:5].copy()
    neighbors[1,1] = 0
    max_neighbor = np.max(neighbors)
    prominence = center_adc - max_neighbor

    #计算cluster的宽度和高度
    threshold = 0.1 * center_adc
    mask = patch > threshold
    rows, cols = np.where(mask)
    if len(rows) == 0:
        width_time = 0
        width_phi = 0
        aspect_ratio = 0
    else:
        width_time = rows.max()-rows.min()+1
        width_phi = cols.max()-cols.min()+1
        aspect_ratio = width_phi / width_time

    return {
        "total_adc": total_adc,
        "max_adc": max_adc,
        "center_adc": center_adc,
        "nonzero_count": nonzero_count,
        "occupancy": occupancy,
        "edge_adc": edge_adc,
        "edge_ratio": edge_ratio,
        "edge_nonzero_count": edge_nonzero_count,
        "edge_occupancy": edge_occupancy,
        "prominence":prominence,
        "width_time":width_time,
        "width_phi":width_phi,
        "aspect_ratio":aspect_ratio
    }

对所有的Cluster进行分类
![png](Type4Cluster.png)

In [ ]:
# Classify all clusters and collect statistics
HIGH_ADC_THRESHOLD = gMaxAdc  #High ADC抓取前5%
print(f"High ADC Threshold = {HIGH_ADC_THRESHOLD:.1f}")

def classify_patch(patch):
    features = patch_features(patch)

    labels = []

    if features["occupancy"] <= 0.15:
        labels.append("Low Occupancy")

    if features["edge_occupancy"] >= 0.50: #周边24个格子，有12个格子有ADC，超过50%
        labels.append("Edge Heavy")

    if features["max_adc"] >= HIGH_ADC_THRESHOLD:
        labels.append("High ADC")

    if features["occupancy"] >= 0.40:
        labels.append("Dense")

    if features["prominence"] <= 10:
        labels.append("Low Prominence")
        
    if len(labels) == 0:
        labels.append("Normal")

    return labels, features


results = []

for n in range(len(df)):

    patch = (
        df.iloc[n, SKIP_COLUMN:matrix_size+SKIP_COLUMN]
        .values
        .reshape(WINDOW_SIZE, WINDOW_SIZE)
    )

    labels, features = classify_patch(patch)

    row = {
        "cluster_id": n,
        "label": ", ".join(labels)
    }

    row.update(features)

    results.append(row)


results_df = pd.DataFrame(results)

results_df.head()

#print(results_df["occupancy"].describe())
#print(results_df["max_adc"].describe())
#print(results_df["edge_ratio"].describe())
results_df["prominence"].describe()

统计cluster的数量

In [ ]:
# Count cluster categories

label_counts = results_df["label"].value_counts()

print(label_counts)

In [ ]:
label_counts.plot(kind="bar", figsize=(8,4))

plt.xlabel("Cluster Category")
plt.ylabel("Count")
plt.title("Cluster Category Counts")

#plt.tight_layout()
plt.show()

定义显示函数

In [ ]:
def show_category_patches(category_name, n_show=6):

    category_df = results_df[
        results_df["label"].str.contains(category_name)
    ]

    print(f"{category_name}: {len(category_df)} clusters")

    fig, axes = plt.subplots(3, 2, figsize=(8, 12))
    axes = axes.flatten()

    for i in range(min(n_show, len(category_df))):

        cluster_id = category_df.iloc[i]["cluster_id"]

        patch = (
            df.iloc[int(cluster_id), SKIP_COLUMN:matrix_size+SKIP_COLUMN]
            .values
            .reshape(WINDOW_SIZE, WINDOW_SIZE)
        )

        im = axes[i].imshow(
            patch,
            origin="lower"
        )
        layer_id = df.iloc[int(cluster_id)]["layer"]
        zelem_id = df.iloc[int(cluster_id)]["zelem"]
        axes[i].set_title(
            f"Cluster {int(cluster_id)}\n"
            f"Layer={layer_id}  Zelem={zelem_id}"
        )

        fig.colorbar(im, ax=axes[i])

    for j in range(min(n_show, len(category_df)), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

显示Low Occupancy

In [ ]:
show_category_patches("Low Occupancy")

显示Edge Heavy

In [ ]:
show_category_patches("Edge Heavy")

显示High ADC

In [ ]:
show_category_patches("High ADC")

显示Dense

In [ ]:
show_category_patches("Dense")

显示prominence >= 10

In [ ]:
show_category_patches("Low Prominence")

显示Normal的

In [ ]:
show_category_patches("Normal")

按分类把所有的cluster保存到当前目录下

In [ ]:
if False:
    from pathlib import Path
    
    output_dir = Path("allcluster_gallery")
    output_dir.mkdir(exist_ok=True)
    
    categories = [
        "Low Occupancy",
        "Edge Heavy",
        "High ADC",
        "Dense",
        "Normal"
    ]
    
    for category in categories:
    
        category_dir = output_dir / category
        category_dir.mkdir(exist_ok=True)
    
        selected = results_df[
            results_df["label"].str.contains(category)
        ]
    
        print(f"{category}: {len(selected)} clusters")
    
        for _, row in selected.iterrows():
    
            cluster_id = int(row["cluster_id"])
            layer_id = int(df.loc[cluster_id, "layer"])
            zelem_id = int(df.loc[cluster_id, "zelem"])
            adc = int(df.loc[cluster_id, "adc"])
    
            patch = (
                df.iloc[cluster_id, SKIP_COLUMN:SKIP_COLUMN+WINDOW_SIZE*WINDOW_SIZE]
                .values
                .reshape(7,7)
            )
    
            fig, ax = plt.subplots(figsize=(4,4))
    
            im = ax.imshow(
                patch,
                origin="lower"
            )
            
            ax.set_title(
                f"Cluster {cluster_id}\n"
                f"Layer={layer_id}  Z={zelem_id}\n"
                f"ADC={adc}"
            )
    
            fig.colorbar(im, ax=ax)
    
            filename = (
                category_dir /
                f"L{layer_id}_Z{zelem_id}_cluster_{cluster_id}.png"
            )
    
            fig.savefig(
                filename,
                dpi=150,
                bbox_inches="tight"
            )
    
            plt.close(fig)
    
    print("Done!")

In [ ]:
#show width_phi vs width_time
table = pd.crosstab(
    results_df["width_time"],
    results_df["width_phi"]
)

plt.figure(figsize=(6,5))

plt.imshow(
    table,
    origin="lower",
    aspect="auto"
)

plt.colorbar(label="Cluster Count")

plt.xticks(range(7), range(1,8))
plt.yticks(range(7), range(1,8))

plt.xlabel("Width in Phi")
plt.ylabel("Width in Time")

plt.show()

In [ ]:
# Peak ADC / Total ADC / Center ADC Histogram

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5)
)

# Peak ADC
axes[0].hist(
    results_df["max_adc"],
    bins=30,
    edgecolor="black"
)
axes[0].set_title("Peak ADC")
axes[0].set_xlabel("Peak ADC")
axes[0].set_ylabel("Number of Clusters")
axes[0].grid(alpha=0.3)

# Total ADC
axes[1].hist(
    results_df["total_adc"],
    bins=30,
    edgecolor="black"
)
axes[1].set_title("Total ADC")
axes[1].set_xlabel("Total ADC")
axes[1].set_ylabel("Number of Clusters")
axes[1].grid(alpha=0.3)

# Center ADC
axes[2].hist(
    results_df["center_adc"],
    bins=30,
    edgecolor="black"
)
axes[2].set_title("Center ADC")
axes[2].set_xlabel("Center ADC")
axes[2].set_ylabel("Number of Clusters")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

把result_df，保存下来

In [ ]:
import os

if os.path.exists("cluster_features.csv"):
    os.remove("cluster_features.csv")
    print("Deleted!")

results_df.to_csv(
    "cluster_features.csv",
    index=False
)

print("Saved!")